In [1]:
class Agent:
    # Class variables — shared across ALL agents
    total_agents_created = 0
    default_model = "llama-3.3-70b"

    def __init__(self, name):
        # Instance variables — unique to each agent
        self.name = name
        Agent.total_agents_created += 1


agent1 = Agent("ResearchBot")
agent2 = Agent("WriterBot")
agent3 = Agent("ReviewBot")

print(Agent.total_agents_created)  # → 3  (shared counter)
print(agent1.name)                  # → ResearchBot  (instance specific)
print(agent1.default_model)         # → llama-3.3-70b  (accessed via instance)
print(Agent.default_model)          # → llama-3.3-70b  (accessed via class)

3
ResearchBot
llama-3.3-70b
llama-3.3-70b


In [ ]:
class Agent:
    total_count = 0

    def __init__(self, name, model):
        self.name = name
        self.model = model
        Agent.total_count += 1

    # Regular method — works with instance data
    def describe(self):
        return f"{self.name} using {self.model}"

    # Class method — works with the class itself, not instance
    @classmethod
    def from_config(cls, config):
        return cls(config["name"], config["model"])

    # Static method — just a function inside the class, no self/cls
    @staticmethod
    def estimate_cost(tokens):
        return tokens * 0.000003

In [4]:
agent = Agent("Bot", "llama-3.3")
agent.describe()    # ✅ works
# Agent.describe()    # ❌ TypeError — needs self

'Bot using llama-3.3'

In [5]:
config = {"name": "ResearchBot", "model": "llama-3.3-70b"}
agent = Agent.from_config(config)    # alternative way to create

In [6]:
cost = Agent.estimate_cost(1000)    # no instance needed
print(cost)

0.003


In [7]:
class Conversation:
    def __init__(self):
        self.messages = []

    @property
    def length(self):
        return len(self.messages)

    @property
    def token_estimate(self):
        total_chars = sum(len(m["content"]) for m in self.messages)
        return total_chars // 4

    def add(self, role, content):
        self.messages.append({"role": role, "content": content})


conv = Conversation()
conv.add("user", "What is Python?")
conv.add("assistant", "Python is a programming language used for...")

# Access like attributes — no parentheses!
print(conv.length)            # → 2
print(conv.token_estimate)    # → 14

2
14


In [8]:
class Agent:
    def __init__(self, name):
        self.name = name
        self.messages = []

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})


class ToolAgent(Agent):
    def __init__(self, name, tools):
        super().__init__(name)        # call parent's __init__
        self.tools = tools

    def list_tools(self):
        return [t["name"] for t in self.tools]

In [9]:
class Agent:
    def think(self):
        return "Generic thinking"


class ResearchAgent(Agent):
    def think(self):
        return "Searching the web..."


class WriterAgent(Agent):
    def think(self):
        # Call parent's version AND add to it
        base = super().think()
        return f"{base} → Drafting content"


print(Agent().think())            # → Generic thinking
print(ResearchAgent().think())    # → Searching the web...
print(WriterAgent().think())      # → Generic thinking → Drafting content

Generic thinking
Searching the web...
Generic thinking → Drafting content


In [10]:
def logger(func):           # takes a function
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Done {func.__name__}")
        return result
    return wrapper          # returns a new function

@logger
def greet(name):
    return f"Hello {name}"

print(greet("Luffy"))

Calling greet
Done greet
Hello Luffy


In [11]:
def repeat(times):              # outer: takes the argument
    def decorator(func):        # middle: takes the function
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator


@repeat(times=3)
def say_hi():
    print("Hi!")

say_hi()    # prints "Hi!" 3 times

Hi!
Hi!
Hi!


In [ ]:
import random

class Agent:

    total_agents = 0
    default_max_tokens = 1024

    def __init__(self, name, model):
        self.name = name
        self.model = model
        self.messages = []
        Agent.total_agents += 1
    
    def add_message(self, role, content):
        self.messages.append(f"role: {role}, content: {content}")

    @property 
    def message_count(self):
        return len(self.messages)
    
    @property 
    def estimated_tokens(self):
        return sum(len(c) for c in self.messages)//4
    
    @classmethod 
    def from_config(cls, config_dict):
        return cls(config_dict["name"], config_dict["model"])

    @staticmethod
    def is_valid_role(role):
        return role in ["user", "assistant", "system", "tool"]

class ResearchAgent(Agent):
    def __init__(self, name, model, sources):
        super().__init__(name, model)
        self.sources = sources

    def add_message(self, role, content):
        super().add_message(role, ["Research"]+content)

class WriterAgent(Agent):
    def __init__(self, name, model, style):
        super().__init__(name, model)
        self.style = style
    
    def add_message(self, role, content):
        return super().add_message(role, ["Writer/style"]+content)


def retry(max_attempts):              # outer: takes the argument
    def decorator(func):        # middle: takes the function
        def wrapper(*args, **kwargs):
            for _ in range(max_attempts):
                result = func(*args, **kwargs)
                if result:
                    return result
                if _==max_attempts and result == ConnectionError:
                    print(f"Failed after {max_attempts} attempts")
                    return None
                
            return result
        return wrapper
    return decorator

@retry(max_attempts=5)
def unstable_call(name):
    if random.random() < 0.7:
        raise ConnectionError("Network failed")
    return f"Success! Got data for {name}"

result = unstable_call("LangChain")
print(result)


agent1 = Agent("ResearchBot", "llama-3.3-70b")
agent1.add_message("user", "Find papers on AI agents")
agent1.add_message("assistant", "I found 5 papers")

agent2 = Agent.from_config({"name": "WriterBot", "model": "llama-3.3-70b"})
agent2.add_message("user", "Write blog")

print(agent1.message_count)        # → 2
print(agent1.estimated_tokens)     # → some number
print(Agent.total_agents)          # → 2
print(Agent.is_valid_role("user")) # → True
print(Agent.is_valid_role("admin"))# → False
        
    
researcher = ResearchAgent("ResearchBot", "llama-3.3", ["web", "arxiv"])
researcher.add_message("user", "Find papers on RAG")
print(researcher.messages)
# → [{"role": "user", "content": "[Research] Find papers on RAG"}]

writer = WriterAgent("WriterBot", "llama-3.3", "casual")
writer.add_message("user", "Write a blog post")
print(writer.messages)
# → [{"role": "user", "content": "[Writer/casual] Write a blog post"}]


ConnectionError: Network failed

In [16]:
import random


class Agent:
    total_agents = 0
    default_max_tokens = 1024

    def __init__(self, name, model):
        self.name = name
        self.model = model
        self.messages = []
        Agent.total_agents += 1

    def add_message(self, role, content):
        self.messages.append({"role": role, "content": content})

    @property
    def message_count(self):
        return len(self.messages)

    @property
    def estimated_tokens(self):
        return sum(len(m["content"]) for m in self.messages) // 4

    @classmethod
    def from_config(cls, config_dict):
        return cls(config_dict["name"], config_dict["model"])

    @staticmethod
    def is_valid_role(role):
        return role in ["user", "assistant", "system", "tool"]


class ResearchAgent(Agent):
    def __init__(self, name, model, sources):
        super().__init__(name, model)
        self.sources = sources

    def add_message(self, role, content):
        super().add_message(role, f"[Research] {content}")


class WriterAgent(Agent):
    def __init__(self, name, model, style):
        super().__init__(name, model)
        self.style = style

    def add_message(self, role, content):
        super().add_message(role, f"[Writer/{self.style}] {content}")


def retry(max_attempts):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt} failed: {e}")
            print(f"Failed after {max_attempts} attempts")
            return None
        return wrapper
    return decorator


@retry(max_attempts=5)
def unstable_call(name):
    if random.random() < 0.7:
        raise ConnectionError("Network failed")
    return f"Success! Got data for {name}"


result = unstable_call("LangChain")
print(result)

agent1 = Agent("ResearchBot", "llama-3.3-70b")
agent1.add_message("user", "Find papers on AI agents")
agent1.add_message("assistant", "I found 5 papers")

agent2 = Agent.from_config({"name": "WriterBot", "model": "llama-3.3-70b"})
agent2.add_message("user", "Write blog")

print(agent1.message_count)
print(agent1.estimated_tokens)
print(Agent.total_agents)
print(Agent.is_valid_role("user"))
print(Agent.is_valid_role("admin"))

researcher = ResearchAgent("ResearchBot", "llama-3.3", ["web", "arxiv"])
researcher.add_message("user", "Find papers on RAG")
print(researcher.messages)

writer = WriterAgent("WriterBot", "llama-3.3", "casual")
writer.add_message("user", "Write a blog post")
print(writer.messages)

Attempt 1 failed: Network failed
Attempt 2 failed: Network failed
Attempt 3 failed: Network failed
Attempt 4 failed: Network failed
Success! Got data for LangChain
2
10
2
True
False
[{'role': 'user', 'content': '[Research] Find papers on RAG'}]
[{'role': 'user', 'content': '[Writer/casual] Write a blog post'}]
